# 01 · Preparar a base documental

## 1. Configuração do projeto

In [1]:
# Localiza a raiz para permitir os imports do projeto.
import sys
from pathlib import Path
from pprint import pprint

root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from src.config import DATA

print("Raiz do projeto:", root)
print("Pasta de dados:", DATA)

Raiz do projeto: /Users/fernandotrinta/Desktop/Infnet
Pasta de dados: /Users/fernandotrinta/Desktop/Infnet/data


## 2. Arquivos do currículo

In [2]:
# O PDF é a fonte da indexação; o TXT é auxiliar nesta demonstração.
curriculo_txt = DATA / "curriculo.txt"
curriculo_pdf = DATA / "curriculo.pdf"

print("Arquivo realmente usado na indexação:", curriculo_pdf)
print("Existe?", curriculo_pdf.is_file())
print("\nArquivo auxiliar usado para gerar o PDF:", curriculo_txt)
print("Existe?", curriculo_txt.is_file())

Arquivo realmente usado na indexação: /Users/fernandotrinta/Desktop/Infnet/data/curriculo.pdf
Existe? True

Arquivo auxiliar usado para gerar o PDF: /Users/fernandotrinta/Desktop/Infnet/data/curriculo.txt
Existe? True


## 3. Criação do PDF

In [3]:
# Gera data/curriculo.pdf a partir do TXT auxiliar.
from scripts.setup_assets import resume_pdf

resume_pdf()

print("PDF criado em:", curriculo_pdf)
print("Existe?", curriculo_pdf.is_file())

PDF criado em: /Users/fernandotrinta/Desktop/Infnet/data/curriculo.pdf
Existe? True


## 4. Carregamento das vagas

In [4]:
# Lê as vagas fictícias de data/vagas_dummy.json.
from src.repository import fixtures

vagas_arquivo = DATA / "vagas_dummy.json"
vagas = fixtures(DATA)

print("Arquivo de vagas:", vagas_arquivo)
print("Quantidade de vagas carregadas:", len(vagas))
print("\nPrimeira vaga:")
pprint(vagas[0].model_dump())

Arquivo de vagas: /Users/fernandotrinta/Desktop/Infnet/data/vagas_dummy.json
Quantidade de vagas carregadas: 5

Primeira vaga:
{'company': 'Aurora Labs',
 'description': 'Worldwide remote, including Brazil. Senior Python, FastAPI '
                'and PostgreSQL. B2 English required. Working hours UTC-3. '
                'Salary: USD 6,500 per month.',
 'id': 'demo-aurora',
 'location': '',
 'salary_hint': '',
 'source': 'Dados fictícios',
 'title': 'Senior Python Backend Engineer',
 'url': 'https://example.com/jobs/aurora'}


## 5. Criação do índice

In [5]:
# Lê o PDF, cria os vetores e salva o índice FAISS.
from src.rag import build_index, INDEX

quantidade_trechos = build_index(vagas, resume=curriculo_pdf)

print("Currículo usado na indexação:", curriculo_pdf)
print("Índice salvo em:", INDEX)
print("Quantidade de trechos indexados:", quantidade_trechos)

/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1112 23:04:42.509000 11198 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 13838.67it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8560.95it/s]

Currículo usado na indexação: /Users/fernandotrinta/Desktop/Infnet/data/curriculo.pdf
Índice salvo em: /Users/fernandotrinta/Desktop/Infnet/data/runtime/faiss_index
Quantidade de trechos indexados: 6


## 6. Arquivos gerados

In [6]:
# Confere os arquivos do índice local.
print("Arquivos gerados no índice:")
for arquivo in sorted(INDEX.iterdir()):
    print("-", arquivo.name)

Arquivos gerados no índice:
- index.faiss
- index.pkl


## 7. Teste de recuperação

In [7]:
# Busca somente documentos marcados como currículo.
from src.rag import load_index, retrieve

load_index.cache_clear()
resultado = retrieve("currículo de Python", k=1, kind="curriculo")
pprint(resultado)

[{'kind': 'curriculo',
  'source': 'curriculo.pdf',
  'trecho': 'Currículo fictício - Alex Silva Senior Backend Engineer / '
            'Desenvolvedor backend sênior Skills: Python, FastAPI, PostgreSQL, '
            'Docker Inglês: B2 Cargo alvo: Senior Backend Engineer'}]


## 8. Modelo NLI

In [8]:
# Carrega o modelo usado para verificar afirmações.
from src.verificador import nli_model

verificador = nli_model()
print("Rótulos do NLI:", verificador.model.config.id2label)

/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 106/106 [00:00<00:00, 9090.27it/s]

Rótulos do NLI: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


## Onde este notebook atende à rubrica

**4.4 — Preparar o índice FAISS.** Nas seções 5 a 7, confira a quantidade de trechos, os arquivos do índice e a recuperação de um trecho do currículo. O [notebook 05](05_ferramentas_rag.ipynb) usa esse índice já construído.